In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import json
import os
import re
from typing import Dict, List, Optional, Tuple

# =========================
# CONFIG
# =========================
EMAIL = "carlos.hernan.suarez@correounivalle.edu.co"
API_KEY = None                         # opcional si tienes key
BASE_URL = "https://api.openalex.org/works"

INTERVENTION_DATE = "2022-11-30"
START_DATE = "2019-08-02"
END_DATE = "2026-03-30"

WORK_TYPES = ["article", "review"]
PER_PAGE = 100   # OpenAlex doc actual: máximo 100
SLEEP_SECONDS = 0.25

SAVE_EVERY_PAGES = 25
MAX_RETRIES = 8
BACKOFF_BASE = 2

TREATMENT_QUERY = (
    '"chatgpt" OR "generative ai" OR "large language model" OR '
    '"large language models" OR llm OR "foundation model" OR '
    '"foundation models" OR "generative artificial intelligence"'
)

CONTROL_QUERY = (
    '"coffee drying" OR "coffee fermentation" OR "coffee roasting" OR '
    '"postharvest coffee" OR "coffee processing" OR '
    '"fermentation of coffee" OR "roasted coffee"'
)

# =========================
# HELPERS
# =========================
def reconstruct_abstract(abstract_inverted_index):
    if not abstract_inverted_index:
        return None

    pos_to_word = {}
    for word, positions in abstract_inverted_index.items():
        for pos in positions:
            pos_to_word[pos] = word

    if not pos_to_word:
        return None

    max_pos = max(pos_to_word.keys())
    tokens = [pos_to_word.get(i, "") for i in range(max_pos + 1)]
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def safe_get(d, path, default=None):
    cur = d
    for p in path:
        if isinstance(cur, dict):
            cur = cur.get(p, default)
        elif isinstance(cur, list) and isinstance(p, int):
            if 0 <= p < len(cur):
                cur = cur[p]
            else:
                return default
        else:
            return default
        if cur is None:
            return default
    return cur


def parse_authorships(authorships):
    if not authorships:
        return "", "", ""

    authors, institutions, countries = [], [], []

    for a in authorships:
        author_name = safe_get(a, ["author", "display_name"], "")
        if author_name:
            authors.append(author_name)

        for inst in (a.get("institutions", []) or []):
            inst_name = inst.get("display_name")
            ccode = inst.get("country_code")
            if inst_name:
                institutions.append(inst_name)
            if ccode:
                countries.append(ccode)

    authors = list(dict.fromkeys(authors))
    institutions = list(dict.fromkeys(institutions))
    countries = list(dict.fromkeys(countries))

    return "; ".join(authors), "; ".join(institutions), "; ".join(countries)


def parse_topics(work):
    topics = work.get("topics", []) or []
    topic_names, field_names, domain_names = [], [], []

    for t in topics:
        tname = t.get("display_name")
        if tname:
            topic_names.append(tname)

        field = t.get("field", {}) or {}
        domain = t.get("domain", {}) or {}

        fname = field.get("display_name")
        dname = domain.get("display_name")

        if fname:
            field_names.append(fname)
        if dname:
            domain_names.append(dname)

    topic_names = list(dict.fromkeys(topic_names))
    field_names = list(dict.fromkeys(field_names))
    domain_names = list(dict.fromkeys(domain_names))

    return "; ".join(topic_names), "; ".join(field_names), "; ".join(domain_names)


def build_params(search_text, cursor="*"):
    filters = [
        f"from_publication_date:{START_DATE}",
        f"to_publication_date:{END_DATE}",
        f"type:{'|'.join(WORK_TYPES)}",
        "is_paratext:false",
        "is_retracted:false",
    ]

    params = {
        "filter": ",".join(filters),
        "search": search_text,
        "per-page": PER_PAGE,
        "cursor": cursor,
        "mailto": EMAIL,
    }

    if API_KEY:
        params["api_key"] = API_KEY

    return params


def request_openalex_with_retry(params, max_retries=MAX_RETRIES):
    for attempt in range(max_retries):
        try:
            r = requests.get(BASE_URL, params=params, timeout=120)

            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                if retry_after is not None:
                    wait = float(retry_after)
                else:
                    wait = BACKOFF_BASE ** attempt

                print(f"429 recibido. Esperando {wait:.1f}s antes de reintentar...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except requests.exceptions.HTTPError as e:
            if getattr(e.response, "status_code", None) == 429:
                wait = BACKOFF_BASE ** attempt
                print(f"HTTP 429. Esperando {wait:.1f}s...")
                time.sleep(wait)
                continue
            raise

        except requests.exceptions.RequestException as e:
            wait = min(BACKOFF_BASE ** attempt, 60)
            print(f"Error de red: {e}. Reintentando en {wait:.1f}s...")
            time.sleep(wait)

    raise RuntimeError("Se agotaron los reintentos contra OpenAlex.")


def extract_record(work, series_label):
    authors_str, institutions_str, countries_str = parse_authorships(work.get("authorships"))
    topic_names, field_names, domain_names = parse_topics(work)
    abstract_text = reconstruct_abstract(work.get("abstract_inverted_index"))

    source_name = safe_get(work, ["primary_location", "source", "display_name"])
    source_type = safe_get(work, ["primary_location", "source", "type"])
    host_org_name = safe_get(work, ["primary_location", "source", "host_organization_name"])

    best_oa_source = safe_get(work, ["best_oa_location", "source", "display_name"])
    best_oa_url = safe_get(work, ["best_oa_location", "landing_page_url"])
    pdf_url = safe_get(work, ["best_oa_location", "pdf_url"])

    ids = work.get("ids", {}) or {}
    doi = ids.get("doi") or work.get("doi")
    pmid = ids.get("pmid")
    pmcid = ids.get("pmcid")

    concepts = work.get("concepts", []) or []
    concept_names = [c.get("display_name") for c in concepts if c.get("display_name")]
    concept_names = list(dict.fromkeys(concept_names))

    keywords = work.get("keywords", []) or []
    keyword_names = []
    for kw in keywords:
        if isinstance(kw, dict):
            name = kw.get("display_name")
            if name:
                keyword_names.append(name)
        elif isinstance(kw, str):
            keyword_names.append(kw)
    keyword_names = list(dict.fromkeys(keyword_names))

    return {
        "series": series_label,
        "openalex_id": work.get("id"),
        "doi": doi,
        "pmid": pmid,
        "pmcid": pmcid,
        "title": work.get("title") or work.get("display_name"),
        "display_name": work.get("display_name"),
        "publication_date": work.get("publication_date"),
        "publication_year": work.get("publication_year"),
        "type": work.get("type"),
        "language": work.get("language"),
        "cited_by_count": work.get("cited_by_count"),
        "is_oa": safe_get(work, ["open_access", "is_oa"]),
        "oa_status": safe_get(work, ["open_access", "oa_status"]),
        "journal": source_name,
        "journal_type": source_type,
        "publisher_or_host_org": host_org_name,
        "best_oa_source": best_oa_source,
        "landing_page_url": best_oa_url,
        "pdf_url": pdf_url,
        "authors": authors_str,
        "institutions": institutions_str,
        "countries": countries_str,
        "topics": topic_names,
        "fields": field_names,
        "domains": domain_names,
        "concepts": "; ".join(concept_names),
        "keywords": "; ".join(keyword_names),
        "abstract": abstract_text,
    }


def deduplicate_works(df):
    out = df.copy()

    if "openalex_id" in out.columns:
        out = out.drop_duplicates(subset=["openalex_id"])

    if "doi" in out.columns:
        has_doi = out["doi"].notna() & (out["doi"].astype(str).str.strip() != "")
        with_doi = out.loc[has_doi].drop_duplicates(subset=["doi"])
        without_doi = out.loc[~has_doi]
        out = pd.concat([with_doi, without_doi], ignore_index=True)

    return out


def save_checkpoint(rows, out_csv, state_path, cursor, page_num, series_label):
    df = pd.DataFrame(rows)
    df = deduplicate_works(df)
    df.to_csv(out_csv, index=False)

    state = {
        "series": series_label,
        "cursor": cursor,
        "page_num": page_num,
        "n_rows": len(df),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(state_path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    print(f"[{series_label}] Checkpoint guardado: {out_csv} | filas={len(df)} | página={page_num}")


def load_existing(out_csv, state_path):
    rows = []
    cursor = "*"
    page_num = 0

    if os.path.exists(out_csv):
        old_df = pd.read_csv(out_csv)
        rows = old_df.to_dict(orient="records")
        print(f"Se cargaron {len(rows)} filas previas desde {out_csv}")

    if os.path.exists(state_path):
        with open(state_path, "r", encoding="utf-8") as f:
            state = json.load(f)
        cursor = state.get("cursor", "*")
        page_num = state.get("page_num", 0)
        print(f"Reanudando desde cursor guardado. Página previa={page_num}")

    return rows, cursor, page_num


def download_series_resume(search_text, series_label, out_csv, state_path, max_records=None):
    rows, cursor, page_num = load_existing(out_csv, state_path)
    seen_ids = set()

    if rows:
        for r in rows:
            oid = r.get("openalex_id")
            if oid:
                seen_ids.add(oid)

    total_expected = None

    while True:
        params = build_params(search_text=search_text, cursor=cursor)
        data = request_openalex_with_retry(params)

        meta = data.get("meta", {}) or {}
        results = data.get("results", []) or []

        if total_expected is None:
            total_expected = meta.get("count")
            print(f"[{series_label}] Total esperado según meta.count: {total_expected}")

        if not results:
            save_checkpoint(rows, out_csv, state_path, cursor, page_num, series_label)
            break

        for work in results:
            oid = work.get("id")
            if oid and oid in seen_ids:
                continue

            rec = extract_record(work, series_label)
            rows.append(rec)

            if oid:
                seen_ids.add(oid)

            if max_records is not None and len(rows) >= max_records:
                save_checkpoint(rows, out_csv, state_path, cursor, page_num, series_label)
                print(f"[{series_label}] Alcanzado max_records={max_records}")
                return pd.DataFrame(rows)

        page_num += 1
        print(f"[{series_label}] Página {page_num} | acumulados: {len(rows)}")

        next_cursor = meta.get("next_cursor")

        if page_num % SAVE_EVERY_PAGES == 0:
            save_checkpoint(rows, out_csv, state_path, next_cursor or cursor, page_num, series_label)

        if not next_cursor:
            save_checkpoint(rows, out_csv, state_path, cursor, page_num, series_label)
            break

        cursor = next_cursor
        time.sleep(SLEEP_SECONDS)

    df = pd.DataFrame(rows)
    df = deduplicate_works(df)
    df.to_csv(out_csv, index=False)
    return df


def add_time_variables(df):
    out = df.copy()
    out["publication_date"] = pd.to_datetime(out["publication_date"], errors="coerce")
    out["ym"] = out["publication_date"].dt.to_period("M").astype(str)
    out["post_chatgpt"] = (out["publication_date"] >= pd.Timestamp(INTERVENTION_DATE)).astype("Int64")
    return out


def build_monthly_counts(df):
    tmp = df.copy()
    tmp["publication_date"] = pd.to_datetime(tmp["publication_date"], errors="coerce")
    tmp = tmp.dropna(subset=["publication_date"])

    monthly = (
        tmp.groupby(["series", pd.Grouper(key="publication_date", freq="MS")])
           .size()
           .reset_index(name="n_docs")
           .rename(columns={"publication_date": "month_start"})
    )
    return monthly


def save_final_outputs(treatment_df, control_df, prefix="openalex_chatgpt_vs_coffee_2019_2026"):
    combined = pd.concat([treatment_df, control_df], ignore_index=True)
    combined = deduplicate_works(combined)
    combined = add_time_variables(combined)

    monthly = build_monthly_counts(combined)

    combined.to_csv(f"{prefix}_combined_clean.csv", index=False)
    monthly.to_csv(f"{prefix}_monthly_counts.csv", index=False)

    # Excel opcional
    with pd.ExcelWriter(f"{prefix}_outputs.xlsx", engine="openpyxl") as writer:
        treatment_df.to_excel(writer, sheet_name="treatment", index=False)
        control_df.to_excel(writer, sheet_name="control", index=False)
        combined.to_excel(writer, sheet_name="combined_clean", index=False)
        monthly.to_excel(writer, sheet_name="monthly_counts", index=False)

    print("\nArchivos finales guardados:")
    print(f"- {prefix}_combined_clean.csv")
    print(f"- {prefix}_monthly_counts.csv")
    print(f"- {prefix}_outputs.xlsx")


def main():
    treatment_csv = "treatment_partial.csv"
    treatment_state = "treatment_state.json"

    control_csv = "control_partial.csv"
    control_state = "control_state.json"

    print("Descargando tratamiento...")
    treatment_df = download_series_resume(
        search_text=TREATMENT_QUERY,
        series_label="treatment",
        out_csv=treatment_csv,
        state_path=treatment_state,
        max_records=None
    )

    print("\nDescargando control café...")
    control_df = download_series_resume(
        search_text=CONTROL_QUERY,
        series_label="control_coffee",
        out_csv=control_csv,
        state_path=control_state,
        max_records=None
    )

    print("\nResumen preliminar:")
    print("Treatment:", treatment_df.shape)
    print("Control:", control_df.shape)

    save_final_outputs(treatment_df, control_df)


if __name__ == "__main__":
    main()

Descargando tratamiento...


/tmp/ipykernel_4159/991604474.py:297: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  old_df = pd.read_csv(out_csv)


Se cargaron 344809 filas previas desde treatment_partial.csv
Reanudando desde cursor guardado. Página previa=3450
[treatment] Total esperado según meta.count: 488370
[treatment] Página 3451 | acumulados: 344909
[treatment] Página 3452 | acumulados: 345009
[treatment] Página 3453 | acumulados: 345109
[treatment] Página 3454 | acumulados: 345209
[treatment] Página 3455 | acumulados: 345309
[treatment] Página 3456 | acumulados: 345409
[treatment] Página 3457 | acumulados: 345509
[treatment] Página 3458 | acumulados: 345609
[treatment] Página 3459 | acumulados: 345709
[treatment] Página 3460 | acumulados: 345809
[treatment] Página 3461 | acumulados: 345909
[treatment] Página 3462 | acumulados: 346009
[treatment] Página 3463 | acumulados: 346109
[treatment] Página 3464 | acumulados: 346209
[treatment] Página 3465 | acumulados: 346309
[treatment] Página 3466 | acumulados: 346409
[treatment] Página 3467 | acumulados: 346509
[treatment] Página 3468 | acumulados: 346609
[treatment] Página 3469 